# 142 — Semantic Caching

**What this workbook teaches:**
- How embedding-based caching differs from exact-match caching
- How cosine similarity works as a "semantic closeness" score
- How the `SemanticCache` class stores and retrieves responses
- How threshold choice (e.g. 0.85 vs 0.95) changes hit/miss behaviour

---

**Two sections:**
- **Part A** — Pure Python, no API key required. Walk through the mechanics with fake vectors.
- **Part B** — Live OpenAI calls. Requires `OPENAI_API_KEY` in your environment.

In [ ]:
%pip install -q openai python-dotenv langgraph

---
## Part A — Concepts (no API key needed)

### What is semantic caching?

**Exact-match caching** stores responses keyed by the exact query string. It only hits when the user types the identical question.

**Semantic caching** stores responses keyed by an *embedding* of the query — a vector of floats that captures meaning. A new query hits the cache if its embedding is close enough (above a similarity threshold) to a cached embedding, even if the wording differs.

**Workflow for each incoming query:**
1. Embed the query → `query_vec` (a list of floats)
2. Compare `query_vec` to every cached embedding using cosine similarity
3. If the best match ≥ threshold → **cache hit**, return stored response (no LLM call)
4. If no match ≥ threshold → **cache miss**, call LLM, store result

**Threshold tuning:**
- Higher threshold (e.g. 0.95) → fewer hits, but responses are very close matches
- Lower threshold (e.g. 0.80) → more hits, but may serve semantically adjacent but imprecise answers
- Default in this example: **0.92**

In [ ]:
# Part A — Cell 2: Cosine similarity with fake vectors
# This is the exact cosine_sim() from src/tools.py.

import math

def cosine_sim(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b + 1e-10)


# Identical vectors → similarity = 1.0
v1 = [0.8, 0.1, 0.5, 0.3]
v2 = [0.8, 0.1, 0.5, 0.3]
print(f"Identical vectors:                 {cosine_sim(v1, v2):.4f}")

# Similar vectors (small perturbation) → high similarity
v3 = [0.81, 0.09, 0.51, 0.29]
print(f"Similar vectors (minor tweak):     {cosine_sim(v1, v3):.4f}")

# Different topic vectors → low similarity
v4 = [0.1, 0.9, 0.0, 0.8]
print(f"Different topic vectors:           {cosine_sim(v1, v4):.4f}")

# Orthogonal vectors → similarity = 0.0
v5 = [0.0, 0.0, 1.0, 0.0]
v6 = [1.0, 0.0, 0.0, 0.0]
print(f"Orthogonal vectors:                {cosine_sim(v5, v6):.4f}")

print()
print("Interpretation: values close to 1.0 = same meaning, close to 0.0 = unrelated.")

In [ ]:
# Part A — Cell 3: Walk through SemanticCache with mock embeddings
# Uses the same class from src/tools.py but with hand-crafted vectors.

from dataclasses import dataclass, field

@dataclass
class SemanticCache:
    threshold: float = 0.92
    _entries: list = field(default_factory=list)
    hits: int = 0
    misses: int = 0

    def get(self, query_vec: list[float]) -> str | None:
        for vec, response in self._entries:
            if cosine_sim(query_vec, vec) >= self.threshold:
                self.hits += 1
                return response
        self.misses += 1
        return None

    def set(self, query_vec: list[float], response: str) -> None:
        self._entries.append((query_vec, response))

    def stats(self) -> dict:
        total = self.hits + self.misses
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": round(self.hits / total, 3) if total else 0,
        }


# Mock embeddings for 3 semantic clusters
# "machine learning" cluster
ml_vec    = [0.9, 0.1, 0.0, 0.1]
ml_vec2   = [0.89, 0.11, 0.01, 0.09]   # near-duplicate — should hit

# "deep learning" cluster  
dl_vec    = [0.7, 0.6, 0.1, 0.0]
dl_vec2   = [0.71, 0.59, 0.09, 0.01]   # near-duplicate — should hit

# "databases" — unrelated
db_vec    = [0.0, 0.0, 0.9, 0.8]

cache = SemanticCache(threshold=0.92)

queries = [
    (ml_vec,  "What is machine learning?",                 "ML is a field of AI that enables computers to learn from data."),
    (ml_vec2, "Can you explain machine learning to me?",   None),   # expect cache hit
    (dl_vec,  "What is deep learning?",                    "Deep learning uses neural networks with many layers."),
    (dl_vec2, "How does deep learning work?",              None),   # expect cache hit
    (db_vec,  "What is a relational database?",            None),   # expect miss
]

print(f"{'Source':<12} {'Query':<45} {'Response (truncated)'}")
print("-" * 90)
for vec, query, stored_response in queries:
    cached = cache.get(vec)
    if cached:
        source = "CACHE HIT"
        answer = cached
    else:
        # Simulate LLM call
        answer = stored_response or f"[LLM answer for: {query[:30]}...]"
        cache.set(vec, answer)
        source = "LLM CALL"
    print(f"{source:<12} {query:<45} {answer[:40]}")

print()
print("Stats:", cache.stats())

In [ ]:
# Part A — Cell 4: Threshold sensitivity demo
# Same mock queries run against threshold=0.85 vs threshold=0.95

def run_demo(threshold: float, query_vecs: list[tuple]) -> dict:
    """Run a cache simulation and return stats."""
    cache = SemanticCache(threshold=threshold)
    results = []
    for i, (query, vec, is_first_in_cluster) in enumerate(query_vecs):
        cached = cache.get(vec)
        if cached:
            results.append((query, "HIT"))
        else:
            cache.set(vec, f"Answer {i}")
            results.append((query, "MISS"))
    return {"stats": cache.stats(), "results": results}


# 6 queries: 3 semantic clusters, each with a near-duplicate
# Similarity within cluster ~0.997; across clusters ~0.3
query_vecs = [
    ("What is machine learning?",           [0.900, 0.100, 0.000, 0.100], True),
    ("Explain machine learning",            [0.895, 0.102, 0.002, 0.098], False),  # sim ~0.9997
    ("What is deep learning?",              [0.700, 0.600, 0.100, 0.000], True),
    ("How does deep learning work?",        [0.698, 0.601, 0.099, 0.001], False),  # sim ~0.9999
    ("What is a relational database?",      [0.000, 0.050, 0.900, 0.800], True),
    ("Explain relational databases",        [0.001, 0.048, 0.901, 0.799], False),  # sim ~0.9999
]

for threshold in [0.85, 0.92, 0.99]:
    out = run_demo(threshold, query_vecs)
    hits   = [q for q, r in out["results"] if r == "HIT"]
    misses = [q for q, r in out["results"] if r == "MISS"]
    print(f"\nThreshold = {threshold}  →  {out['stats']}")
    print(f"  Hits:   {hits}")
    print(f"  Misses: {misses}")

---
## Part B — Live OpenAI calls

**Requires:** `OPENAI_API_KEY` in your `.env` file or environment.

Runs 5 queries through the LangGraph workflow. Near-duplicate queries will hit the semantic cache instead of calling the LLM, and you'll see the hit rate and LLM call savings.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. "
        "Add it to your .env file or export it before running Part B."
    )
print("OPENAI_API_KEY loaded.")

In [ ]:
# Part B — Run the semantic caching workflow with 5 queries
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), "."))

from openai import OpenAI
from src.workflow import create_workflow

QUERIES = [
    "What is machine learning?",
    "Can you explain machine learning to me?",   # near-duplicate → expect cache hit
    "What is deep learning?",
    "How does deep learning work?",              # near-duplicate → expect cache hit
    "What is a relational database?",            # new topic → expect miss
]

client = OpenAI(api_key=OPENAI_API_KEY)
workflow = create_workflow()
config = {"configurable": {"client": client}}

result = workflow.invoke({
    "queries": QUERIES,
    "threshold": 0.92,
    "responses": [],
    "cache_stats": {},
}, config=config)

print("=== Semantic Cache Demo ===\n")
llm_calls = 0
for r in result["responses"]:
    label = "[CACHE HIT]" if r["source"] == "cache" else f"[LLM {r['latency_ms']}ms]"
    if r["source"] == "llm":
        llm_calls += 1
    print(f"{label} {r['query']}")
    print(f"  {r['response'][:80]}...\n")

stats = result["cache_stats"]
print(f"Hit rate:      {stats['hit_rate'] * 100:.0f}%")
print(f"LLM calls:     {llm_calls} of {len(QUERIES)} queries")
print(f"LLM calls saved: {len(QUERIES) - llm_calls}")
print(f"Full stats:    {json.dumps(stats, indent=2)}")

### Semantic vs Exact-Match Caching

| Property | Exact-match cache | Semantic cache |
|---|---|---|
| Cache key | Full prompt string (hash) | Embedding vector |
| Matches | Identical prompts only | Similar-meaning prompts |
| False positives | Zero | Possible (threshold too low) |
| Storage | Dict / Redis | Vector store (in-memory or hosted) |
| Lookup cost | O(1) hash lookup | O(n) similarity scan or ANN index |
| Best for | Repeated identical queries | User paraphrases, typos, synonyms |

Semantic caching trades a small risk of false positives for dramatically higher hit rates
in real applications where users rarely type the same query twice.

### Cache Eviction Strategies

A semantic cache without eviction grows unbounded. Three common strategies:

| Strategy | How it works | Best for |
|---|---|---|
| **LRU** (Least Recently Used) | Evict the entry not accessed longest | General chat (topics shift over time) |
| **TTL** (Time-to-Live) | Evict entries older than N seconds | Frequently updated knowledge |
| **Capacity** | Evict when len > max_size | Memory-constrained deployments |
| **Score decay** | Weight old hits less; evict below threshold | Long-running production caches |

Combining TTL (freshness) + LRU (relevance) is the most common production approach.

### Embedding Model Comparison

| Model | Dimensions | Cost / MTok | Speed | Best for |
|---|---|---|---|---|
| `text-embedding-3-small` | 1536 | $0.02 | Fast | Most use cases |
| `text-embedding-3-large` | 3072 | $0.13 | Moderate | High-precision semantic search |
| `text-embedding-ada-002` | 1536 | $0.10 | Fast | Legacy (prefer 3-small) |
| `all-MiniLM-L6-v2` (local) | 384 | Free | Very fast | Offline, privacy-sensitive |

> For a semantic cache, `text-embedding-3-small` is the sweet spot: cheap, fast, and accurate enough
> that the embedding cost is negligible compared to the LLM calls it avoids.

### Exercise 1 — Add Hit/Miss Counters

Modify the `SemanticCache` class to track `hit_count` and `miss_count` as instance attributes.
Print a summary after 5 queries.

In [ ]:
# Exercise 1: Add hit_count and miss_count to SemanticCache.
# Increment hit_count in lookup() when similarity >= threshold.
# Increment miss_count when no match found.
# Add a hit_rate property: hit_count / (hit_count + miss_count).
#
# TODO: implement SemanticCacheV2 with these additions.


In [ ]:
# Exercise 1 — Answer Key
import math

def cosine_sim(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(x*x for x in b))
    return dot / (na * nb + 1e-8)

class SemanticCacheV2:
    def __init__(self, threshold=0.92):
        self.store = []  # list of (embedding, answer) tuples
        self.threshold = threshold
        self.hit_count = 0   # <-- new
        self.miss_count = 0  # <-- new

    def lookup(self, embedding):
        for stored_emb, answer in self.store:
            if cosine_sim(embedding, stored_emb) >= self.threshold:
                self.hit_count += 1  # <-- increment on hit
                return answer
        self.miss_count += 1  # <-- increment on miss
        return None

    def add(self, embedding, answer):
        self.store.append((embedding, answer))

    @property
    def hit_rate(self):
        total = self.hit_count + self.miss_count
        return self.hit_count / total if total else 0.0

# Demo with mock embeddings
cache = SemanticCacheV2(threshold=0.92)
mock_emb_a = [1.0, 0.0, 0.0]
cache.add(mock_emb_a, 'Paris is the capital of France.')

queries = [
    ([1.0, 0.0, 0.0],    'exact match'),
    ([0.99, 0.1, 0.0],   'near match'),
    ([0.7, 0.7, 0.0],    'partial match'),
    ([0.0, 0.0, 1.0],    'no match'),
    ([0.98, 0.15, 0.05], 'near match 2'),
]
for emb, label in queries:
    result = cache.lookup(emb)
    print(f'  {label:15}: {"HIT" if result else "MISS"}')

print(f'\nHit rate: {cache.hit_rate:.0%} ({cache.hit_count}/{cache.hit_count+cache.miss_count})')
# Tracking hit_count/miss_count lets you tune the threshold based on production data.

### Exercise 2 — Add TTL Expiry

Extend `SemanticCacheV2` so that entries expire after `ttl_seconds` seconds.
A lookup should return None for expired entries.

In [ ]:
# Exercise 2: Add a ttl_seconds parameter to SemanticCacheV2.__init__().
# Store the insertion timestamp alongside each entry.
# In lookup(), skip entries older than ttl_seconds.
#
# TODO: implement SemanticCacheV3(threshold=0.92, ttl_seconds=300)


In [ ]:
# Exercise 2 — Answer Key
import time

class SemanticCacheV3(SemanticCacheV2):
    def __init__(self, threshold=0.92, ttl_seconds=300):
        super().__init__(threshold)
        self.ttl_seconds = ttl_seconds
        self.store = []  # now stores (embedding, answer, timestamp)

    def add(self, embedding, answer):
        self.store.append((embedding, answer, time.time()))  # record insertion time

    def lookup(self, embedding):
        now = time.time()
        for stored_emb, answer, ts in self.store:
            if now - ts > self.ttl_seconds:  # expired — skip it
                continue
            if cosine_sim(embedding, stored_emb) >= self.threshold:
                self.hit_count += 1
                return answer
        self.miss_count += 1
        return None

# Demo: short TTL to show expiry
cache3 = SemanticCacheV3(threshold=0.90, ttl_seconds=0.01)  # 10ms TTL
cache3.add([1.0, 0.0, 0.0], 'Fresh answer')
print('Immediate lookup:', cache3.lookup([1.0, 0.0, 0.0]))  # HIT
time.sleep(0.02)
print('After TTL expired:', cache3.lookup([1.0, 0.0, 0.0]))  # MISS (expired)
# TTL prevents stale answers from being served — critical for time-sensitive knowledge.

In [ ]:
# Part A: 5-query cache simulation with near-duplicates
# Shows realistic hit/miss behavior without any API calls
import math

def cosine_sim(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    return dot / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(x*x for x in b)) + 1e-8)

# Simulate a conversation: 5 queries, 2 are near-duplicates of query 1
MOCK_QUERIES = [
    {'text': 'What is the capital of France?',          'emb': [1.0, 0.0, 0.0, 0.0]},
    {'text': 'Tell me the capital city of France.',      'emb': [0.97, 0.15, 0.05, 0.0]},  # near-dup
    {'text': 'What is the largest city in Germany?',    'emb': [0.2, 0.9, 0.1, 0.0]},
    {'text': "France's capital — what city is it?",     'emb': [0.96, 0.18, 0.08, 0.0]},  # near-dup
    {'text': 'What is the population of Tokyo?',        'emb': [0.1, 0.1, 0.95, 0.0]},
]

cache = []  # list of (embedding, answer)
THRESHOLD = 0.92
llm_calls = 0

for q in MOCK_QUERIES:
    hit = next((ans for emb, ans in cache if cosine_sim(q['emb'], emb) >= THRESHOLD), None)
    if hit:
        status = 'HIT '
        answer = hit
    else:
        status = 'MISS'
        answer = f'[LLM answered: {q["text"][:30]}...]'
        cache.append((q['emb'], answer))
        llm_calls += 1
    print(f'{status} | {q["text"][:45]:<45} | {answer[:40]}')

print(f'\nLLM calls: {llm_calls}/5 | Hit rate: {(5-llm_calls)/5:.0%}')

### Production Deployment Patterns

In production, semantic caches sit in front of the LLM API:

```
User query
    │
    ▼
Embed query  ──►  vector similarity scan  ──► HIT?  ──► return cached answer
                                              │ MISS
                                              ▼
                                       LLM API call
                                              │
                                              ▼
                               store (embedding, answer) in cache
                                              │
                                              ▼
                                       return answer
```

**Production backends:**
- **Redis + RediSearch** — low-latency, persistent, horizontal scaling
- **Qdrant / Milvus** — dedicated vector DB, HNSW index for O(log n) lookup
- **In-memory (this example)** — fine for ≤ 10k entries, lost on restart

In [ ]:
# Threshold sensitivity analysis: how does threshold affect hit rate vs false-positive rate?

# Pairs: (query, cached_question, true_similarity, is_semantically_equivalent)
TEST_PAIRS = [
    ('What is the capital of France?', 'What is the capital of France?', 1.00, True),
    ('What is the capital of France?', 'Tell me France\'s capital city.', 0.96, True),
    ('What is the capital of France?', 'What is the largest city in France?', 0.88, False),  # trap!
    ('What is the capital of France?', 'What is the capital of Germany?', 0.82, False),
    ('What is the capital of France?', 'Recommend a restaurant in Paris.', 0.60, False),
]

print(f'{'Threshold':>10} | {'Hits':>4} | {'False+':>6} | {'Precision':>10}')
print('-' * 42)
for thresh in [0.80, 0.85, 0.90, 0.92, 0.95, 0.99]:
    hits = [(p[3], p[3]) for p in TEST_PAIRS if p[2] >= thresh]
    tp = sum(1 for actual, _ in hits if actual)
    fp = sum(1 for actual, _ in hits if not actual)
    precision = tp / (tp + fp) if (tp + fp) else 1.0
    print(f'{thresh:>10.2f} | {tp+fp:>4} | {fp:>6} | {precision:>10.0%}')

print()
print('Recommendation: 0.92 gives full precision (no false positives) while catching near-duplicates.')

### Cache Warming

A cold cache has 0% hit rate on day one. Three strategies to pre-warm it:

1. **Replay logs** — embed your top-100 historical queries and seed the cache before launch
2. **FAQ seeding** — for known-question systems (support chatbot, docs assistant), pre-embed all FAQs
3. **Synthetic generation** — use an LLM to generate paraphrase variants of common questions

Even seeding 50–100 entries dramatically improves day-one hit rates for narrow-domain applications.

In [ ]:
# Summary stats helper — useful for tuning in production
def cache_stats(log: list[dict]) -> dict:
    """Compute summary stats from a list of {'hit': bool} log entries."""
    total = len(log)
    hits = sum(1 for e in log if e.get('hit'))
    return {
        'total_queries': total,
        'cache_hits': hits,
        'cache_misses': total - hits,
        'hit_rate': round(hits / total, 3) if total else 0,
        'llm_calls_saved': hits,
        'estimated_savings_usd': round(hits * 0.001, 4),  # assume $0.001/LLM call average
    }

# Simulate a production log
import random
random.seed(42)
# After warm-up, ~35% hit rate is realistic for a domain-specific assistant
mock_log = [{'hit': random.random() < 0.35} for _ in range(100)]

stats = cache_stats(mock_log)
for k, v in stats.items():
    print(f'  {k}: {v}')

---
## What's Next

- **GPTCache** (github.com/zilliztech/GPTCache) — production semantic cache library with Redis + Milvus backends
- **LangChain semantic cache** — `from langchain.cache import InMemorySemanticCache` — drop-in integration
- **Example 141 — Prompt Caching:** cache at the system-prompt level (Anthropic API feature) for a different layer of cost savings
- **Example 143 — Context Compression:** reduce context *before* caching to lower embedding cost
- **ANN indexes (HNSW, FAISS):** replace linear scan O(n) with approximate nearest-neighbor O(log n) for large caches